# Secure distributed Computation Final Project
## Luke Holmes and Jake Pappas

We are going to implement linear regression inference on secret inputs to provide disease progression inferences to clients securely. We will use a 1-server, n-clients architecture, using OT to generate multiplication triples. Our dataset is the scikit-learn diabetes dataset.

In [ ]:
# Useful imports and utility functions
import pychor
import galois
import pandas as pd
import sklearn
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

server = pychor.Party('server')
user = pychor.Party('user')

import numpy as np

GF = galois.GF(2**61-1)

@pychor.local_function
def share(x):
    s1 = GF.Random()
    s2 = GF(x) - s1
    return s1, s2

In [ ]:
# Load as a pandas dataframe
diabetes_df = server.constant(load_diabetes(as_frame=True))
df = server.constant(diabetes_df.val.frame)
model = LinearRegression()
df_test = df.val.iloc[:2]
model.fit(df.val.drop(columns='target'), df.val['target'])
scaled_coefs = model.coef_ * 10**8



print(f"test data: {df_test}")
print(f"Coefficents:{scaled_coefs}")
print(f"Intercept:{model.intercept_}")

In [ ]:
# @dataclass
# class patient:
#     # age sex bmi bp s1 s2 s3 s4 s5 s6
#     age: float
#     sex: float
#     bmi: float
#     bp: float
#     s1: float
#     s2: float
#     s3: float
#     s4: float
#     s5: float
#     s6: float

# patient1 = patient(0.4, 0.4, 0.3, 0.2, 0.2, 0.6, 0.2, 0.7, 0.8, 0.4)

# @pychor.local_function
gen_triples(10)
def infer():
    age = SecInt.input(1)
    sex = SecInt.input(2)
    bmi = SecInt.input(2)
    bp = SecInt.input(1)
    s1 = SecInt.input(1)
    s2 = SecInt.input(2)
    s3 = SecInt.input(1)
    s4 = SecInt.input(3)
    s5 = SecInt.input(4)
    s6 = SecInt.input(1)

    age_w = SecInt.input(server.constant(scaled_coefs[0]))
    sex_w = SecInt.input(server.constant(scaled_coefs[1]))
    bmi_w = SecInt.input(server.constant(scaled_coefs[2]))
    bp_w = SecInt.input(server.constant(scaled_coefs[3]))
    s1_w = SecInt.input(server.constant(scaled_coefs[4]))
    s2_w = SecInt.input(server.constant(scaled_coefs[5]))
    s3_w = SecInt.input(server.constant(scaled_coefs[6]))
    s4_w = SecInt.input(server.constant(scaled_coefs[7]))
    s5_w = SecInt.input(server.constant(scaled_coefs[8]))
    s6_w = SecInt.input(server.constant(scaled_coefs[9]))

    total = 0
    total += age * age_w
    total += sex * sex_w
    total += bmi * bmi_w
    total += bp * bp_w
    total += s1 * s1_w
    total += s2 * s2_w
    total += s3 * s3_w
    total += s4 * s4_w
    total += s5 * s5_w
    total += s6 * s6_w

    return total

print(infer())

In [ ]:
from dataclasses import dataclass

multiplication_triples = []

@dataclass
class SecInt:
    # s1 is server's share of the value, and s2 is user's share
    s1: galois.GF
    s2: galois.GF

    @classmethod
    def input(cls, val):
        """Secret share an input: server holds s1, and user holds s2"""
        s1, s2 = share(val).untup(2)
        if server in val.parties:
            s2.send(server, user)
            return SecInt(s1, s2)
        else:
            s1.send(user, server)
            return SecInt(s1, s2)

    def __add__(x, y):
        """Add two SecInt objects using local addition of shares"""
        return SecInt(x.s1 + y.s1, x.s2 + y.s2)

    def __mul__(x, y):
        """Multiply two SecInt objects using a triple"""
        triple = multiplication_triples.pop()
        r1, r2 = protocol_mult((x.s1, x.s2), (y.s1, y.s2), triple)
        return SecInt(r1, r2)

    def reveal(self):
        """Reveal the secret value by broadcast and reconstruction"""
        self.s1.send(server, user)
        self.s2.send(user, server)
        return self.s1 + self.s2

def functionality_gen_triple():
    Fgen = pychor.Party('Fgen')

    def deal_shares(x):
        s1, s2 = share(x).untup(2)
        s1.send(Fgen, server)
        s2.send(Fgen, user)
        return s1, s2

    # Step 1: generate a, b, c
    a = Fgen.constant(GF.Random())
    b = Fgen.constant(GF.Random())
    c = a * b

    # Step 2: secret share a, b, c
    a1, a2 = deal_shares(a)
    b1, b2 = deal_shares(b)
    c1, c2 = deal_shares(c)
    return (a1, a2), (b1, b2), (c1, c2)

def protocol_mult(x, y, triple):
    x1, x2 = x
    y1, y2 = y
    (a1, a2), (b1, b2), (c1, c2) = triple

    # Step 1. server computes d_1 = x_1 - a_1 and sends the result to user
    d1 = x1 - a1
    d1.send(server, user)

    # Step 2. user computes d_2 = x_2 - a_2 and sends the result to server
    d2 = x2 - a2
    d2.send(user, server)

    # Step 3: server and user both compute $d = d_1 + d_2 = x - a$
    d = d1 + d2

    # Step 4. server computes e_1 = y_1 - b_1 and sends the result to user
    e1 = y1 - b1
    e1.send(server, user)

    # Step 5. user computes e_2 = y_2 - b_2 and sends the result to server
    e2 = y2 - b2
    e2.send(user, server)

    # Step 6. server and user both compute $e = e_1 + e_2 = y - b$
    e = e1 + e2

    # Step 7. server computes r_1 = d*e + d*b_1 + e*a_1 + c_1
    r_1 = d * e + d * b1 + e * a1 + c1

    # Step 8. user computes r_2 = 0 + d*b_2 + e*a_2 + c_2
    r_2 = d * b2 + e * a2 + c2

    return r_1, r_2

def gen_triples(n):
    for _ in range(n):
        multiplication_triples.append(functionality_gen_triple())